In [ ]:
# HookLink2iOS — última IPA bem-sucedida, seleção pelo conteúdo, 64 MiB por bloco
REPO="joao9vegeto-stack/KintTany"
BRANCH="HookLink2iOS"
TITLE="0400486024DEC000"
PROJECT_DRIVE_FOLDER="1IynIGg57WnKydeqsdVzOp90tPkiX_eNY"
CHUNK=64*1024*1024
assert CHUNK == 64*1024*1024 and CHUNK%(256*1024)==0
import os,time,re,json,io,hashlib,zipfile,plistlib,shutil,getpass
from pathlib import Path
from urllib.parse import urlparse
import requests
from google.colab import auth,userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from tqdm.auto import tqdm
from datetime import datetime
STEP=0
TOTAL_STEPS=10
def stage(text):
    global STEP
    STEP+=1
    print(f"\n{'='*66}\n[{STEP:02d}/{TOTAL_STEPS}] {text}\n{'='*66}",flush=True)
def progress_hash(path,algo="sha256"):
    path=Path(path)
    h=hashlib.new(algo)
    total=path.stat().st_size
    with tqdm(total=total,unit="B",unit_scale=True,unit_divisor=1024,
              desc=f"Hash {algo.upper()}: {path.name[:22]}",dynamic_ncols=True,mininterval=.5) as bar:
        with path.open("rb") as f:
            while True:
                block=f.read(8*1024*1024)
                if not block:break
                h.update(block);bar.update(len(block))
    return h.hexdigest()

import google.auth
ROOT=Path("/content/HookLink2iOS-transfer");ROOT.mkdir(exist_ok=True)
stage("Preparar diretório e transferência de 64 MiB")
print("Diretório:",ROOT,"| Tamanho de bloco:",CHUNK//(1024*1024),"MiB",flush=True)
stage("Autenticar GitHub / localizar Actions")
try:TOKEN=userdata.get("GH_TOKEN") or ""
except Exception:TOKEN=""
if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions:read para localizar a última IPA): ").strip()
if not TOKEN:raise RuntimeError("GH_TOKEN é obrigatório para localizar artefatos neste repositório")
HEAD={"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
def gitget(path,**opts):
    headers=dict(HEAD)
    if TOKEN:headers["Authorization"]="Bearer "+TOKEN
    r=requests.get("https://api.github.com/repos/"+REPO+path,headers=headers,timeout=40,**opts)
    if r.status_code in (401,403):raise RuntimeError("Configure Colab Secrets GH_TOKEN com permissão Actions read.")
    r.raise_for_status();return r.json()
_HASH_CACHE={}
def sha(path,algo="sha256"):
    p=Path(path)
    stat=p.stat()
    cache_key=(str(p.resolve()),algo,stat.st_size,stat.st_mtime_ns)
    if cache_key not in _HASH_CACHE:
        _HASH_CACHE[cache_key]=progress_hash(p,algo)
    return _HASH_CACHE[cache_key]
stage("Autenticar Google Drive")
auth.authenticate_user()
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",credentials=creds,cache_discovery=False)
stage("Localizar a IPA realmente mais recente da branch")
# Sem nomes de workflow, artefato, release, motor ou ABI fixos.
# Ordenamos pela data de publicação do artefato, inclusive em reexecuções.
def pages(path,key,**params):
    page=1
    while True:
        batch=gitget(path,params={**params,"per_page":100,"page":page}).get(key,[])
        yield from batch
        if len(batch)<100:return
        page+=1

def signed_url(art_id):
    rr=requests.get(f"https://api.github.com/repos/{REPO}/actions/artifacts/{art_id}/zip",
                    headers={**HEAD,"Authorization":"Bearer "+TOKEN},allow_redirects=False,timeout=40)
    if rr.status_code not in (301,302,303,307,308) or not rr.headers.get("Location"):
        raise RuntimeError(f"Artefato {art_id}: GitHub HTTP {rr.status_code}, sem URL temporária")
    url=rr.headers["Location"]
    if urlparse(url).scheme!="https":raise RuntimeError("URL de download não usa HTTPS")
    return url

class RemoteZip(io.RawIOBase):
    # zipfile lê o índice central sem transferir a IPA inteira.
    def __init__(self,art):
        self.art=art;self.total=int(art["size_in_bytes"]);self.pos=0
        self.url=signed_url(art["id"]);self.cache={}
    def readable(self):return True
    def seekable(self):return True
    def tell(self):return self.pos
    def seek(self,offset,whence=0):
        self.pos=offset if whence==0 else self.pos+offset if whence==1 else self.total+offset
        if self.pos<0:raise OSError("seek negativo")
        return self.pos
    def read(self,n=-1):
        if n is None or n<0:n=self.total-self.pos
        n=max(0,min(n,self.total-self.pos));out=bytearray()
        while n:
            blocksize=1024*1024
            start=self.pos//blocksize*blocksize
            end=min(start+blocksize,self.total)-1
            if start not in self.cache:
                for attempt in range(5):
                    try:
                        with requests.get(self.url,headers={"Range":f"bytes={start}-{end}","Accept-Encoding":"identity"},
                                          stream=True,timeout=(30,90)) as resp:
                            m=re.fullmatch(r"bytes\s+(\d+)-(\d+)/(\d+)",resp.headers.get("Content-Range",""))
                            if resp.status_code!=206 or not m or tuple(map(int,m.groups()))!=(start,end,self.total):
                                raise RuntimeError(f"Índice ZIP: Range inválido HTTP {resp.status_code}")
                            data=resp.content
                            if len(data)!=end-start+1:raise RuntimeError("Range incompleto")
                        if len(self.cache)>=8:self.cache.pop(next(iter(self.cache)))
                        self.cache[start]=data
                        break
                    except (requests.RequestException,RuntimeError):
                        if attempt==4:raise
                        time.sleep(min(8,2**attempt));self.url=signed_url(self.art["id"])
            off=self.pos-start;part=self.cache[start][off:off+n]
            out.extend(part);self.pos+=len(part);n-=len(part)
        return bytes(out)

def ipa_members(z):
    return [i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith(".ipa")]

def artifact_key(a):return (a.get("created_at") or "",int(a["id"]))

def select_latest(runs,list_artifacts,inspect):
    best=None
    for run in sorted(runs,key=lambda r:(r.get("updated_at") or r.get("created_at") or "",int(r["id"])),reverse=True):
        # updated_at é posterior à publicação dos artefatos em runs concluídos.
        if best and run.get("updated_at") and run["updated_at"]<best[0][0]:break
        for art in sorted(list_artifacts(run),key=artifact_key,reverse=True):
            if art.get("expired") or int(art.get("size_in_bytes",0))<=0:continue
            wr=art.get("workflow_run") or {}
            if wr.get("id",run["id"])!=run["id"] or wr.get("head_branch",BRANCH)!=BRANCH:continue
            if wr.get("head_sha",run["head_sha"])!=run["head_sha"]:continue
            key=artifact_key(art)
            if best and key<best[0]:break
            members=inspect(art)
            if not members:continue
            if len(members)!=1:raise RuntimeError(f"Artefato {art['id']} contém {len(members)} IPAs; seleção ambígua.")
            best=(key,run,art,members[0])
    if best is None:raise RuntimeError("Nenhum artefato não expirado contendo IPA encontrado em builds bem-sucedidos da branch.")
    return best[1:]

def inspect_artifact(art):
    print(f"Inspecionando ZIP {art['id']}: {art['name']} ({art.get('created_at')})",flush=True)
    with RemoteZip(art) as f,zipfile.ZipFile(f) as z:
        return [i.filename for i in ipa_members(z)]

def test_selection():
    def r(i,t):return {"id":i,"head_sha":str(i),"updated_at":t}
    def a(i,t,expired=False):return {"id":i,"created_at":t,"size_in_bytes":1,"expired":expired}
    runs=[r(1,"2026-01-04"),r(2,"2026-01-05"),r(3,"2026-01-06")]
    arts={1:[a(10,"2026-01-03")],2:[a(20,"2026-01-05")],3:[a(30,"2026-01-06"),a(31,"2026-01-06",True)]}
    # Mais recente é diagnóstico; artefato expirado não serve; o workflow intermediário contém IPA.
    assert select_latest(runs,lambda r:arts[r["id"]],lambda a:["arbitrary.ipa"] if a["id"] in (10,20,31) else [])[1]["id"]==20
test_selection()

runs=[r for r in pages("/actions/runs","workflow_runs",branch=BRANCH,status="success")
      if r.get("head_branch")==BRANCH and r.get("status")=="completed" and r.get("conclusion")=="success"]
chosenrun,artifact,selected_member=select_latest(runs,
    lambda run:pages(f"/actions/runs/{run['id']}/artifacts","artifacts"),inspect_artifact)
print(f"IPA MAIS RECENTE: {selected_member}\nRun: {chosenrun['id']} | commit: {chosenrun['head_sha']}\n"
      f"Artefato: {artifact['name']} (ID {artifact['id']}) | publicado: {artifact['created_at']}",flush=True)


def download_actions(art):
    meta=gitget("/actions/artifacts/"+str(art["id"]))
    if meta.get("expired"):raise RuntimeError("Artefato GitHub expirado.")
    if meta.get("name")!=art["name"] or int((meta.get("workflow_run") or {}).get("id",chosenrun["id"]))!=int(chosenrun["id"]):
        raise RuntimeError("Metadados do artefato não correspondem ao run selecionado.")
    digest=meta.get("digest") or ""
    if not re.fullmatch(r"sha256:[0-9a-f]{64}",digest):
        raise RuntimeError("Sem SHA-256 confiável para o artefato GitHub.")
    total=int(meta["size_in_bytes"])
    partial=ROOT/f"artifact-{art['id']}.zip.part"
    finished=ROOT/f"artifact-{art['id']}.zip"
    marker=ROOT/f"artifact-{art['id']}.meta.json"
    identity={"id":art["id"],"bytes":total,"digest":digest}
    if finished.exists():
        if finished.stat().st_size==total and sha(finished)==digest.split(":",1)[1]:
            print("Arquivo completo já existente: SHA-256 OK. Download dispensado.",flush=True)
            return finished
        finished.unlink()
    if not marker.exists() or json.loads(marker.read_text())!=identity:
        partial.unlink(missing_ok=True)
        marker.write_text(json.dumps(identity))
    pos=partial.stat().st_size if partial.exists() else 0
    if pos>total:partial.unlink();pos=0
    if pos:print(f"Retomando em {pos/1024**2:.1f} MiB de {total/1024**2:.1f} MiB",flush=True)
    with tqdm(total=total,initial=pos,unit="B",unit_scale=True,unit_divisor=1024,
              desc="GitHub → Colab | 64 MiB",dynamic_ncols=True,mininterval=.5) as bar:
        while pos<total:
            end=min(total,pos+CHUNK)-1
            bar.set_postfix_str(f"chunk {pos//CHUNK+1}/{(total+CHUNK-1)//CHUNK}")
            for attempt in range(8):
                received=0
                try:
                    hdr={**HEAD,"Authorization":"Bearer "+TOKEN}
                    rr=requests.get(f"https://api.github.com/repos/{REPO}/actions/artifacts/{art['id']}/zip",
                                    headers=hdr,allow_redirects=False,timeout=40)
                    if rr.status_code not in (301,302,303,307,308) or not rr.headers.get("Location"):
                        raise RuntimeError(f"GitHub não forneceu URL do artefato: HTTP {rr.status_code}")
                    signed=rr.headers["Location"]
                    if urlparse(signed).scheme!="https":raise RuntimeError("URL de download não usa HTTPS")
                    with requests.get(signed,headers={"Range":f"bytes={pos}-{end}"},
                                      stream=True,timeout=(30,180)) as resp:
                        m=re.fullmatch(r"bytes\s+(\d+)-(\d+)/(\d+)",resp.headers.get("Content-Range",""))
                        if resp.status_code!=206 or not m or tuple(map(int,m.groups()))!=(pos,end,total):
                            raise RuntimeError(f"Range inválido: HTTP {resp.status_code}, Content-Range={resp.headers.get('Content-Range')!r}")
                        with partial.open("r+b" if partial.exists() else "w+b") as f:
                            f.truncate(pos);f.seek(pos)
                            for block in resp.iter_content(1024*1024):
                                if block:
                                    f.write(block);received+=len(block);bar.update(len(block))
                        if received!=end-pos+1:raise RuntimeError(f"Chunk incompleto: {received} / {end-pos+1} bytes")
                    pos=end+1
                    break
                except Exception as exc:
                    bar.update(-received)
                    if partial.exists():
                        with partial.open("r+b") as f:f.truncate(pos)
                    tqdm.write(f"Chunk {pos//CHUNK+1}: tentativa {attempt+1}/8 falhou: {type(exc).__name__}: {exc}")
                    if attempt==7:raise
                    time.sleep(min(30,2**attempt))
    partial.replace(finished)
    return finished

stage("Baixar artefato GitHub em chunks HTTP Range de 64 MiB")
SOURCE=download_actions(artifact)
origin=f"Run {chosenrun['id']} / artefato {artifact['id']}"
stage("Verificar SHA-256 completo do arquivo recebido")
expected_digest=(artifact.get("digest") or gitget("/actions/artifacts/"+str(artifact["id"])).get("digest") or "").split(":",1)[-1]
assert SOURCE.stat().st_size==int(artifact["size_in_bytes"]),"Tamanho do artefato ZIP divergente"
if sha(SOURCE)!=expected_digest:raise RuntimeError("SHA-256 do artefato GitHub não confere")
print("GitHub ZIP: SHA-256 OK",flush=True)

stage("Inspecionar artefato e extrair IPA com progresso contínuo")
with zipfile.ZipFile(SOURCE) as z:
    nested=[info for info in z.infolist() if info.filename.lower().endswith(".ipa") and not info.is_dir()]
    reports=[info for info in z.infolist() if Path(info.filename).name=="build-report.json"]
    if len(nested)!=1 or nested[0].filename!=selected_member or len(reports)>1:
        raise RuntimeError(f"Conteúdo do artefato diverge da seleção: IPAs={len(nested)}, relatórios={len(reports)}")
    info=nested[0]
    report=json.loads(z.read(reports[0])) if reports else {}
    if report.get("run_id") is not None and str(report["run_id"])!=str(chosenrun["id"]):
        raise RuntimeError("Relatório pertence a outro run")
    if report.get("commit") and report["commit"]!=chosenrun["head_sha"]:
        raise RuntimeError("Commit no relatório não corresponde ao run")
    ENGINE=str(report.get("engine_commit") or "unknown")
    RELEASE=str(report.get("hooklink_release") or "unknown")
    IPA=ROOT/f"{artifact['id']}"/Path(info.filename).name
    IPA.parent.mkdir(exist_ok=True)
    wanted=(report.get("ipa") or {}).get("sha256")
    if wanted is not None and not re.fullmatch(r"[0-9a-fA-F]{64}",str(wanted)):
        raise RuntimeError("SHA-256 inválido no relatório")
    if (report.get("ipa") or {}).get("size") is not None and int(report["ipa"]["size"])!=info.file_size:
        raise RuntimeError("IPA diverge do tamanho declarado pelo build")

    if wanted and IPA.exists() and IPA.stat().st_size==info.file_size and sha(IPA)==wanted.lower():
        print("IPA já extraída e validada; extração dispensada.",flush=True)
    else:
        tmp=IPA.with_suffix(".ipa.part")
        with z.open(info) as src,tmp.open("wb") as dst:
            with tqdm(total=info.file_size,unit="B",unit_scale=True,unit_divisor=1024,
                      desc="Extração da IPA",dynamic_ncols=True,mininterval=.5) as bar:
                while True:
                    block=src.read(8*1024*1024)
                    if not block:break
                    dst.write(block);bar.update(len(block))
        tmp.replace(IPA)
stage("Validar SHA-256, RomFS, executável, bibliotecas Metal e CRC")
if IPA.stat().st_size>=4_000_000_000:raise RuntimeError("IPA excede o limite seguro de 4 GB")
IPA_HASH=sha(IPA)
if wanted and IPA_HASH!=wanted.lower():raise RuntimeError("SHA-256 IPA divergente do build-report.json")
with zipfile.ZipFile(IPA) as z:
    names=z.namelist()
    apps=[name[:-len("Info.plist")] for name in names if re.fullmatch(r"Payload/[^/]+\.app/Info\.plist",name)]
    if len(apps)!=1 or len(names)!=len(set(names)):
        raise RuntimeError("IPA inválida: app principal ausente, ambígua ou entradas duplicadas")
    pre=apps[0]
    plist=plistlib.loads(z.read(pre+"Info.plist"))
    executable=plist.get("CFBundleExecutable")
    if not executable or "/" in executable or pre+executable not in names:
        raise RuntimeError("Executável principal ausente")
    if not any(re.fullmatch(re.escape(pre)+r"msl/b-[^/]+\.img(?:\.[0-9]+)?",entry) for entry in names):
        raise RuntimeError("Bibliotecas Metal ausentes")
    if pre+"romfs.id" not in names:raise RuntimeError("Marcador RomFS ausente")
    embedded=z.read(pre+"romfs.id").decode("utf8").strip()
    if not embedded.startswith("DMROMFSID1 "):raise RuntimeError("Marcador RomFS inválido")
    actual=dict(pair.split("=",1) for pair in embedded.split()[1:] if "=" in pair)
    if actual.get("title","").upper()!=TITLE or not re.fullmatch(r"[0-9a-fA-F]+",actual.get("index","")) or int(actual.get("size","0"))<=0:
        raise RuntimeError("RomFS não corresponde ao título HookLink2iOS")
    declared=(report.get("ipa") or {}).get("contents",{}).get("romfs_marker")
    if declared and embedded!=declared.strip():raise RuntimeError("Marcador RomFS diverge do relatório")
    target=(report.get("romfs") or {}).get("identity")
    if target:
        expected=dict(pair.split("=",1) for pair in target.split()[1:] if "=" in pair)
        for key in ("title","size","index"):
            if actual.get(key)!=expected.get(key):raise RuntimeError(f"RomFS {key} diverge do relatório")
    if (report.get("romfs") or {}).get("size") is not None and int(actual["size"])!=int(report["romfs"]["size"]):
        raise RuntimeError("Tamanho RomFS diverge do relatório")
    if any(entry.lower().endswith("romfs.img") for entry in names):raise RuntimeError("RomFS deveria estar fora da IPA")
    print("Motor:",ENGINE,"| HookLink:",RELEASE,"| RomFS:",actual["title"],actual["size"],actual["index"],flush=True)

    infos=[info for info in z.infolist() if not info.is_dir()]
    total_uncompressed=sum(info.file_size for info in infos)
    with tqdm(total=total_uncompressed,unit="B",unit_scale=True,unit_divisor=1024,
              desc="CRC de todos os arquivos",dynamic_ncols=True,mininterval=.5) as bar:
        for info in infos:
            bar.set_postfix_str(Path(info.filename).name[:28])
            with z.open(info) as f:
                while True:
                    block=f.read(8*1024*1024)
                    if not block:break
                    bar.update(len(block))
print("VALIDAÇÃO CONCLUÍDA: IPA intacta, RomFS externa, versão detectada automaticamente.",flush=True)
print("Arquivo:",IPA,"| tamanho:",IPA.stat().st_size,"| SHA-256:",IPA_HASH,flush=True)


# ETAPA 10: upload Google Drive resumable 64 MiB e verificação de integridade.
stage("Preparar pasta HookLink2iOS/ipa e upload resumível de 64 MiB")
def drive_folder():
    parent=PROJECT_DRIVE_FOLDER
    try:
        drive.files().get(fileId=parent,fields="id",supportsAllDrives=True).execute()
        print("Pasta do projeto encontrada no Drive.",flush=True)
    except Exception:
        matches=drive.files().list(q="name='HookLink2iOS' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            fields="files(id)",pageSize=20).execute().get("files",[])
        parent=matches[0]["id"] if matches else drive.files().create(body={
            "name":"HookLink2iOS","mimeType":"application/vnd.google-apps.folder"},fields="id").execute()["id"]
    entries=drive.files().list(q=f"'{parent}' in parents and name='ipa' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields="files(id)",pageSize=20).execute().get("files",[])
    fid=entries[0]["id"] if entries else drive.files().create(body={
        "name":"ipa","mimeType":"application/vnd.google-apps.folder","parents":[parent]},fields="id").execute()["id"]
    print("Destino Drive /HookLink2iOS/ipa:",fid,flush=True)
    return fid
folder=drive_folder()
name=IPA.stem+"-"+IPA_HASH[:12]+".ipa"
queryname=name.replace("'","\\'")
existing=drive.files().list(q=f"'{folder}' in parents and name='{queryname}' and trashed=false",
    fields="files(id,name,size,md5Checksum,appProperties,webViewLink)",pageSize=100).execute().get("files",[])
print("Conferindo MD5 local, em progresso...",flush=True)
MD5=sha(IPA,"md5")
current=next((x for x in existing if int(x.get("size",0))==IPA.stat().st_size
    and x.get("md5Checksum")==MD5 and (x.get("appProperties") or {}).get("sha256")==IPA_HASH),None)
if current:
    print("IPA idêntica já no Drive. Upload dispensado.",flush=True)
else:
    upload=MediaFileUpload(str(IPA),mimetype="application/octet-stream",chunksize=CHUNK,resumable=True)
    request=drive.files().create(body={"name":name,"parents":[folder],
        "appProperties":{"sha256":IPA_HASH,"titleId":TITLE,"branch":BRANCH,"engine":ENGINE,"hooklink":RELEASE,"runId":str(chosenrun["id"]),"artifactId":str(artifact["id"])}},
        media_body=upload,fields="id,name,size,md5Checksum,appProperties,webViewLink")
    current=None;uploaded=0;total=IPA.stat().st_size
    with tqdm(total=total,unit="B",unit_scale=True,unit_divisor=1024,
              desc="Colab → Drive | 64 MiB",dynamic_ncols=True,mininterval=.5) as bar:
        while current is None:
            for attempt in range(8):
                try:
                    status,current=request.next_chunk()
                    pos=total if current is not None else int(status.resumable_progress) if status else uploaded
                    if pos>uploaded:bar.update(pos-uploaded)
                    uploaded=max(uploaded,pos)
                    bar.set_postfix_str(f"chunk {(uploaded+CHUNK-1)//CHUNK}/{(total+CHUNK-1)//CHUNK}")
                    break
                except Exception as exc:
                    tqdm.write(f"Upload: tentativa {attempt+1}/8 falhou: {type(exc).__name__}: {exc}")
                    if attempt==7:raise
                    time.sleep(min(30,2**attempt))
stage("Conferir upload: tamanho, MD5 e SHA-256 do Drive")
print("Verificando tamanho, MD5 e SHA-256 no Drive...",flush=True)
check=drive.files().get(fileId=current["id"],fields="id,name,size,md5Checksum,appProperties,webViewLink",
    supportsAllDrives=True).execute()
if int(check.get("size",0))!=IPA.stat().st_size or check.get("md5Checksum")!=MD5 or (check.get("appProperties") or {}).get("sha256")!=IPA_HASH:
    raise RuntimeError("Falha de readback Drive: tamanho/MD5/SHA-256 divergentes")
print("TRANSFERÊNCIA FINALIZADA E VALIDADA.",flush=True)
print("Origem:",origin,"| Nome:",check["name"],flush=True)
print("SHA-256:",IPA_HASH,flush=True)
print("Drive:",check.get("webViewLink") or "https://drive.google.com/file/d/"+check["id"]+"/view",flush=True)
